
# Power Transformation (Box-Cox and Yeo-Johnson)

## 1. Introduction to Power Transformation

Power Transformation is a feature transformation technique used in Machine Learning to make data more normally distributed (closer to a Gaussian/Normal distribution).

### Why do we use Power Transformation?

- To reduce skewness (right-skewed or left-skewed data).
- To make data closer to a normal distribution.
- To reduce the influence of extreme values.
- To improve the performance of algorithms that work better with normally distributed data.

In Scikit-learn, we use `PowerTransformer()`.

There are two methods:

1. Box-Cox Transformation
2. Yeo-Johnson Transformation

---

## 2. Box-Cox Transformation

**Definition:** Box-Cox is a power transformation technique that transforms data to make it more normally distributed and reduce skewness.

### Important Condition

Box-Cox works only with strictly positive values.

$$X > 0$$

It cannot directly handle zero or negative values.

### Why do we add 0.0001?

Suppose our dataset contains:

| Original Value |
|---:|
| 0 |
| 2 |
| 5 |
| 10 |

Box-Cox cannot handle zero values.

Therefore, we add a very small positive number (epsilon), such as `0.0001`, to every value.

```python
X_train_transformed = pt.fit_transform(X_train + 0.0001)

X_test_transformed = pt.transform(X_test + 0.0001)
```

After adding epsilon:

| Original Value | After Adding 0.0001 |
|---:|---:|
| 0 | 0.0001 |
| 2 | 2.0001 |
| 5 | 5.0001 |
| 10 | 10.0001 |

**Why do we do this?**

- Box-Cox requires strictly positive values.
- Zero is not allowed.
- Adding a small positive number makes zero positive.
- It allows Box-Cox to perform the transformation.

**Important:** Adding 0.0001 does not solve negative values. Box-Cox cannot directly handle negative values.

### Box-Cox Mathematical Formula

$$
X^{(\lambda)} =
\begin{cases}
\frac{X^\lambda-1}{\lambda}, & \lambda \neq 0 \\[8pt]
\ln(X), & \lambda = 0
\end{cases}
$$

Where:

- $X$ = Original feature value.
- $\lambda$ = Power parameter (automatically calculated).
- $X^{(\lambda)}$ = Transformed value.

The algorithm automatically selects the optimal lambda value to make the data closer to a normal distribution.

### Python Implementation

```python
from sklearn.preprocessing import PowerTransformer
import pandas as pd

# Applying Box-Cox Transformation

pt = PowerTransformer(method='box-cox')

X_train_transformed = pt.fit_transform(X_train + 0.0001)

X_test_transformed = pt.transform(X_test + 0.0001)

# Display lambda values

pd.DataFrame({
    'cols': X_train.columns,
    'box_cox_lambdas': pt.lambdas_
})
```

### Important Parameters

- `fit_transform()` : Learns lambda values from training data and transforms it.
- `transform()` : Uses already learned lambda values on testing data.
- `lambdas_` : Stores the calculated lambda values for each feature.

---

## 3. Yeo-Johnson Transformation

**Definition:** Yeo-Johnson is another power transformation technique used to make data more normally distributed and reduce skewness.

### Biggest Advantage

Unlike Box-Cox, Yeo-Johnson can handle:

- Positive values
- Zero values
- Negative values

**No need to add 0.0001!**

### Example

| Original Value |
|---:|
| -5 |
| 0 |
| 2 |
| 10 |

Yeo-Johnson can directly transform all these values.

### Yeo-Johnson Mathematical Formula

$$
X^{(\lambda)} =
\begin{cases}
\frac{(X+1)^\lambda-1}{\lambda},
& X \geq 0,\ \lambda \neq 0 \\[8pt]

\ln(X+1),
& X \geq 0,\ \lambda = 0 \\[8pt]

-\frac{(1-X)^{2-\lambda}-1}{2-\lambda},
& X < 0,\ \lambda \neq 2 \\[8pt]

-\ln(1-X),
& X < 0,\ \lambda = 2
\end{cases}
$$

Where:

- $X$ = Original feature value.
- $\lambda$ = Automatically calculated power parameter.
- $X^{(\lambda)}$ = Transformed value.

Yeo-Johnson uses separate formulas for positive and negative values.

### Python Implementation

```python
from sklearn.preprocessing import PowerTransformer
import pandas as pd

# Applying Yeo-Johnson Transformation

pt = PowerTransformer(method='yeo-johnson')

X_train_transformed = pt.fit_transform(X_train)

X_test_transformed = pt.transform(X_test)

# Display lambda values

pd.DataFrame({
    'cols': X_train.columns,
    'yeo_johnson_lambdas': pt.lambdas_
})
```

---

## 4. Difference Between Box-Cox and Yeo-Johnson

| Feature | Box-Cox | Yeo-Johnson |
|---|---|---|
| Positive values | Yes | Yes |
| Zero values | No | Yes |
| Negative values | No | Yes |
| Requires adding epsilon for zero | Yes | No |
| Handles skewness | Yes | Yes |
| Calculates lambda automatically | Yes | Yes |

### Why is Yeo-Johnson More Flexible?

1. It can handle negative values.
2. It can handle zero values.
3. No need to add a small constant.
4. It works with a wider range of datasets.

**Important:** Yeo-Johnson is more flexible, but it does not always produce better results than Box-Cox. Both should be evaluated on the dataset.

---

## 5. PowerTransformer vs FunctionTransformer

These are two different transformers in Scikit-learn.

### A. PowerTransformer

**Definition:** PowerTransformer automatically applies a mathematical power transformation to reduce skewness and make data closer to a normal distribution.

It automatically calculates the best lambda value.

Two methods are available:

```python
PowerTransformer(method='box-cox')
```

```python
PowerTransformer(method='yeo-johnson')
```

### When to Use PowerTransformer?

- When data is highly skewed.
- When you want to reduce skewness.
- When you want to make data closer to a normal distribution.
- When you want lambda to be automatically calculated.

### B. FunctionTransformer

**Definition:** FunctionTransformer is used when we want to apply our own mathematical function to the data.

Unlike PowerTransformer, it does not automatically calculate lambda.

We specify the function ourselves.

### Example 1: Log Transformation

```python
from sklearn.preprocessing import FunctionTransformer
import numpy as np

ft = FunctionTransformer(np.log1p)

X_train_transformed = ft.fit_transform(X_train)

X_test_transformed = ft.transform(X_test)
```

Mathematical Formula:

$$f(X) = \log(1+X)$$

`np.log1p(X)` means $\log(1+X)$.

It is useful for reducing right skewness, especially when data contains zero but no values less than -1.

### Example 2: Square Root Transformation

```python
ft = FunctionTransformer(np.sqrt)

X_train_transformed = ft.fit_transform(X_train)
```

Mathematical Formula:

$$f(X) = \sqrt{X}$$

Useful for reducing right skewness in non-negative data.

### Example 3: Custom Function

```python
def square(x):
    return x**2

ft = FunctionTransformer(square)

X_train_transformed = ft.fit_transform(X_train)
```

Mathematical Formula:

$$f(X) = X^2$$

You can apply your own mathematical function using FunctionTransformer.

---

## 6. PowerTransformer vs FunctionTransformer

| Feature | PowerTransformer | FunctionTransformer |
|---|---|---|
| Purpose | Reduce skewness | Apply custom function |
| Lambda calculation | Automatic | Not automatic |
| Available methods | Box-Cox, Yeo-Johnson | Log, square root, square, etc. |
| Custom function | No direct custom function | Yes |
| Normality improvement | Specifically designed for it | Depends on chosen function |

---

## 7. Most Important Exam Revision

### Remember These 5 Points

1. Box-Cox only accepts strictly positive values.
2. Add `0.0001` to handle zero values in Box-Cox, but it cannot fix negative values.
3. Yeo-Johnson handles positive, zero and negative values.
4. PowerTransformer automatically finds the optimal lambda.
5. FunctionTransformer applies the mathematical function that we specify.

### Quick Comparison

| Situation | Transformer |
|---|---|
| Data contains only positive values | Box-Cox or Yeo-Johnson |
| Data contains zero values | Yeo-Johnson |
| Data contains negative values | Yeo-Johnson |
| Automatically find optimal power | PowerTransformer |
| Apply log transformation | FunctionTransformer |
| Apply square root transformation | FunctionTransformer |
| Apply custom mathematical function | FunctionTransformer |
